# An ETL pipeline for idealista18

An **ETL** (extract, transform, load) pipeline moves data from its sources into a store where others can use it: it **extracts** the data from the source formats, **transforms** it into one consistent, documented, checked table, and **loads** that table where its users can read it. The users of this one could be an analyst, a dashboard or a valuation model, so nothing in it depends on any particular model.

The line between ETL and the preprocessing done for a model is one question: *does the step learn anything from the data, or depend on how the data will be split for a model?* If it doesn't, it can be ETL; if it does, it belongs to the modelling pipeline, after the split. The [README](README.md#where-etl-ends-and-modelling-begins) develops the criterion with examples from this project.

The pipeline below produces a single table of every listing of the three cities: descriptive names and explicit types, coded columns decoded, each listing's neighbourhood added from the polygons, and a report of every quality rule it breaks. It is written as a Parquet file.

## Extract

Each source object is read with the same function. The download repeats the one of [part 1](source_and_rda_format.ipynb) so that the pipeline runs on its own: a pipeline must be able to start from nothing.

In [1]:
from pathlib import Path

DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw_rda"
OUTPUT_PARQUET = DATA_DIR / "idealista18_listings.parquet"

CITIES = ("Madrid", "Barcelona", "Valencia")
SOURCE_URL_TEMPLATE = "https://github.com/paezha/idealista18/raw/master/data/{file_name}"

In [2]:
import urllib.request
import warnings

import rdata


def read_idealista18_object(city: str, kind: str) -> object:
    path = RAW_DIR / f"{city}_{kind}.rda"
    if not path.exists():
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(SOURCE_URL_TEMPLATE.format(file_name=path.name), path)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Missing constructor for R class")
        converted_objects = rdata.read_rda(path, default_encoding="utf8")
    return converted_objects[f"{city}_{kind}"]


listings_by_city = {city: read_idealista18_object(city, "Sale") for city in CITIES}
polygons_by_city = {city: read_idealista18_object(city, "Polygons") for city in CITIES}
pois_by_city = {city: read_idealista18_object(city, "POIS") for city in CITIES}

### Checking the source against its documentation

Documentation describes what the data should be; only the data says what it is. Before transforming anything, check the facts the pipeline relies on. First, the row counts:

In [3]:
import pandas as pd

pd.Series({city: len(listings) for city, listings in listings_by_city.items()}, name="rows")

Madrid       94815
Barcelona    61486
Valencia     33622
Name: rows, dtype: int64

They match the documentation. Each city has the four quarter codes, with uneven numbers of listings per quarter:

In [4]:
pd.DataFrame({city: listings["PERIOD"].value_counts().sort_index()
              for city, listings in listings_by_city.items()})

,Madrid,Barcelona,Valencia
PERIOD,,,
201803,21920,17826,9305
201806,12652,7951,4655
201809,15973,12375,5644
201812,44270,23334,14018


Not every column is shared by the three cities:

In [5]:
column_sets = {city: set(listings.columns) for city, listings in listings_by_city.items()}
shared_columns = set.intersection(*column_sets.values())
print(f"{len(shared_columns)} shared columns")
{city: sorted(columns - shared_columns) for city, columns in column_sets.items()}

41 shared columns


{'Madrid': ['DISTANCE_TO_CASTELLANA'],
 'Barcelona': ['DISTANCE_TO_DIAGONAL'],
 'Valencia': ['DISTANCE_TO_BLASCO']}

Each city has a distance to a different landmark: Paseo de la Castellana in Madrid, Avinguda Diagonal in Barcelona, Avenida de Blasco Ibáñez in Valencia.

The documentation defines `UNITPRICE` as the price per square metre. If it was computed from `PRICE` after the anonymisation noise, the two must agree exactly:

In [6]:
pd.Series({city: (listings["UNITPRICE"] - listings["PRICE"] / listings["CONSTRUCTEDAREA"]).abs().max()
           for city, listings in listings_by_city.items()}, name="largest difference")

Madrid       6.366463e-12
Barcelona    6.366463e-12
Valencia     5.456968e-12
Name: largest difference, dtype: float64

The differences are floating-point rounding: `UNITPRICE` is exactly `PRICE / CONSTRUCTEDAREA`, a column derived from the price. That is worth recording in the table's documentation, because anyone who predicts the price must never use it as an input.

## Transform

### Descriptive names

The source names are upper-case abbreviations (`CADASTRALQUALITYID`, `FLOORCLEAN`). The pipeline gives every column a descriptive `snake_case` name, with the unit in the name where there is one (`_eur`, `_m2`, `_km`).

The three landmark distances keep three separate names. They could be merged into one column, "distance to the landmark", but they measure distances to three different places: a merged column would look like one variable while meaning three. An ETL must not hide a difference of meaning behind a shared name, so each city's landmark column is empty in the other two cities.

In [7]:
SHARED_COLUMN_NAMES = {
    "ASSETID": "asset_id",
    "PERIOD": "period",
    "PRICE": "price_eur",
    "UNITPRICE": "unit_price_eur_m2",
    "CONSTRUCTEDAREA": "built_area_m2",
    "ROOMNUMBER": "bedrooms",
    "BATHNUMBER": "bathrooms",
    "HASTERRACE": "has_terrace",
    "HASLIFT": "has_lift",
    "HASAIRCONDITIONING": "has_air_conditioning",
    "HASPARKINGSPACE": "has_parking_space",
    "ISPARKINGSPACEINCLUDEDINPRICE": "parking_included_in_price",
    "PARKINGSPACEPRICE": "parking_space_price_eur",
    "HASNORTHORIENTATION": "faces_north",
    "HASSOUTHORIENTATION": "faces_south",
    "HASEASTORIENTATION": "faces_east",
    "HASWESTORIENTATION": "faces_west",
    "HASBOXROOM": "has_box_room",
    "HASWARDROBE": "has_wardrobe",
    "HASSWIMMINGPOOL": "has_swimming_pool",
    "HASDOORMAN": "has_doorman",
    "HASGARDEN": "has_garden",
    "ISDUPLEX": "is_duplex",
    "ISSTUDIO": "is_studio",
    "ISINTOPFLOOR": "is_top_floor",
    "CONSTRUCTIONYEAR": "construction_year",
    "FLOORCLEAN": "floor",
    "CADCONSTRUCTIONYEAR": "cadastre_construction_year",
    "CADMAXBUILDINGFLOOR": "cadastre_building_floors",
    "CADDWELLINGCOUNT": "cadastre_dwellings_in_building",
    "CADASTRALQUALITYID": "cadastre_quality",
    "DISTANCE_TO_CITY_CENTER": "dist_city_centre_km",
    "DISTANCE_TO_METRO": "dist_metro_km",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
}
LANDMARK_COLUMN_NAMES = {
    "Madrid": ("DISTANCE_TO_CASTELLANA", "dist_castellana_km"),
    "Barcelona": ("DISTANCE_TO_DIAGONAL", "dist_diagonal_km"),
    "Valencia": ("DISTANCE_TO_BLASCO", "dist_blasco_km"),
}

Four source columns are not in the list: three are decoded next, and `geometry` duplicates the coordinates ([part 1](source_and_rda_format.ipynb#reading-an-rda-file-from-python)).

### Decoding coded columns

Some columns store a category as a number or spread it over several columns. Turning them back into named categories is a typical ETL step: no information is added or lost, but the table becomes readable without the documentation at hand.

The condition of the property is spread over three 0/1 columns, `BUILTTYPEID_1` to `BUILTTYPEID_3` (new development, second hand to restore, second hand in good condition). That is one categorical variable in *one-hot* form, which holds only if exactly one of the three is 1 in every row. That is checked before relying on it:

In [8]:
CONDITION_COLUMNS = {"BUILTTYPEID_1": "new_development", "BUILTTYPEID_2": "to_restore", "BUILTTYPEID_3": "good"}

pd.Series({city: listings[list(CONDITION_COLUMNS)].sum(axis=1).value_counts().to_dict()
           for city, listings in listings_by_city.items()}, name="ones per row: rows")

Madrid       {1: 94815}
Barcelona    {1: 61486}
Valencia     {1: 33622}
Name: ones per row: rows, dtype: object

Every row has exactly one 1, so the three columns collapse into one `condition` column without loss. Two more columns are codes with documented meanings: `AMENITYID` (1 = no kitchen equipment and no furniture, 2 = kitchen equipment, 3 = kitchen equipment and furniture) and `FLATLOCATIONID` (1 = exterior, 2 = interior, missing in some rows). A fixed list of categories (`CategoricalDtype`) makes every city's column share the same categories, so they stay categorical when the cities are combined.

In [9]:
CONDITION = pd.CategoricalDtype(list(CONDITION_COLUMNS.values()))
FURNISHING = pd.CategoricalDtype(["none", "kitchen", "kitchen_and_furniture"])
OUTLOOK = pd.CategoricalDtype(["exterior", "interior"])


def decode_coded_columns(listings: pd.DataFrame) -> pd.DataFrame:
    condition_codes = listings[list(CONDITION_COLUMNS)].to_numpy().argmax(axis=1)
    return pd.DataFrame({
        "condition": pd.Categorical.from_codes(condition_codes, dtype=CONDITION),
        "furnishing": listings["AMENITYID"].map({1: "none", 2: "kitchen", 3: "kitchen_and_furniture"}).astype(FURNISHING),
        "outlook": listings["FLATLOCATIONID"].map({1: "exterior", 2: "interior"}).astype(OUTLOOK),
    }, index=listings.index)

### One table per city

Each city's table gets the descriptive names, the decoded columns, its landmark distance and a `city` column. The identifier arrives as an R factor (a pandas category), but an identifier is not a category: it has as many values as there are listings and nothing is ever counted by it. It is stored as a string.

In [10]:
def to_clean_schema(listings: pd.DataFrame, city: str) -> pd.DataFrame:
    source_landmark_column, landmark_column = LANDMARK_COLUMN_NAMES[city]
    city_table = listings[list(SHARED_COLUMN_NAMES)].rename(columns=SHARED_COLUMN_NAMES)
    city_table["asset_id"] = city_table["asset_id"].astype("str")
    city_table = pd.concat([city_table, decode_coded_columns(listings)], axis="columns")
    city_table[landmark_column] = listings[source_landmark_column]
    city_table["city"] = city.lower()
    return city_table


to_clean_schema(listings_by_city["Valencia"], "Valencia").head(3).T

,1,2,3
asset_id,A9651535568269959084,A13801394133398327300,A8013965313053471366
period,201812,201812,201812
price_eur,111000.0,169000.0,162000.0
unit_price_eur_m2,1480.0,1320.3125,1883.72093
built_area_m2,75,128,86
bedrooms,2,4,2
bathrooms,1,2,1
has_terrace,1,0,0
has_lift,1,1,1
has_air_conditioning,1,1,1


### Combining the cities

The three tables are stacked into one. Their columns differ only in the landmark distances, which `concat` aligns by name, filling each city's missing landmark columns with `NaN`. `ignore_index=True` builds a new index: each city's index restarts at 1 (R's row numbers), and keeping them would repeat every label up to three times.

In [11]:
listings = pd.concat([to_clean_schema(listings_by_city[city], city) for city in CITIES], ignore_index=True)
listings = listings[["city", *listings.columns.drop("city")]]
print(f"{len(listings):,} rows, {listings.shape[1]} columns")
listings.dtypes.astype(str).value_counts()

189,923 rows, 42 columns


Int32       27
float64      9
category     3
str          2
Float64      1
Name: count, dtype: int64

In [12]:
listings.groupby("city", sort=False)[["dist_castellana_km", "dist_diagonal_km", "dist_blasco_km"]].count()

,dist_castellana_km,dist_diagonal_km,dist_blasco_km
city,,,
madrid,94815,0,0
barcelona,0,61486,0
valencia,0,0,33622


Each landmark column is filled for its own city and empty elsewhere, as intended.

### Enrichment: the neighbourhood of each listing

**Enrichment** <!-- TODO: is it the same as feature engineering? -->adds information that the source rows don't carry, from reference data. Here, each listing gets the neighbourhood whose polygon contains it: a *point-in-polygon* join. It needs only the listing's own coordinates and a fixed reference map, so it learns nothing from the listings and belongs in the ETL.

[shapely](https://shapely.readthedocs.io) builds the geometries from the converted polygons, and a spatial index (`STRtree`) avoids testing each of 190,000 points against each of 277 polygons: it first discards the polygons whose bounding box can't contain the point.

In [13]:
import numpy as np
import shapely


def to_shapely_multipolygon(polygons: list[list[np.ndarray]]) -> shapely.MultiPolygon:
    return shapely.MultiPolygon([shapely.Polygon(rings[0], holes=rings[1:]) for rings in polygons])


neighbourhoods = pd.concat(
    [polygons_by_city[city].assign(city=city.lower()) for city in CITIES], ignore_index=True)
neighbourhood_shapes = [to_shapely_multipolygon(polygons) for polygons in neighbourhoods["geometry"]]
print(f"{len(neighbourhood_shapes)} neighbourhoods, all valid: {all(shapely.is_valid(neighbourhood_shapes))}")

277 neighbourhoods, all valid: True


In [14]:
listing_points = shapely.points(listings["longitude"].to_numpy(), listings["latitude"].to_numpy())
spatial_index = shapely.STRtree(neighbourhood_shapes)
point_positions, neighbourhood_positions = spatial_index.query(listing_points, predicate="intersects")

matches_per_listing = np.bincount(point_positions, minlength=len(listings))
pd.Series(matches_per_listing).value_counts().rename_axis("neighbourhoods containing the listing")

neighbourhoods containing the listing
1    189618
0       285
2        20
Name: count, dtype: int64

The predicate is `intersects` rather than `within`, so that a listing exactly on a border still counts as inside. Almost every listing falls inside exactly one neighbourhood, but not all. A few fall inside two: they lie on a shared border, or where two polygons overlap slightly. The idealista18 authors met the same problem when they anonymised the coordinates, and took the neighbourhood with the lowest identifier; the pipeline follows their rule. A few hundred fall inside none. How far are they from the nearest neighbourhood?

In [15]:
METRES_PER_DEGREE = 111_000  # one degree of latitude; a degree of longitude is shorter, so this overestimates

unmatched_positions = np.flatnonzero(matches_per_listing == 0)
_, gap_degrees = spatial_index.query_nearest(listing_points[unmatched_positions], return_distance=True,
                                             all_matches=False)
gap_metres = pd.Series(gap_degrees * METRES_PER_DEGREE, index=listings["city"].iloc[unmatched_positions].to_numpy())
gap_metres.groupby(level=0, sort=False).describe(percentiles=[0.5, 0.9])[["count", "50%", "90%", "max"]].round(0)

,count,50%,90%,max
madrid,94.0,1.0,249.0,396248.0
barcelona,147.0,15.0,34.0,57.0
valencia,44.0,17.0,37.0,50.0


Most are within a few tens of metres of the nearest polygon (the distances are in metres, and slightly overestimated). The polygons don't tile the city perfectly: they leave thin gaps between neighbours, and a listing in a gap belongs, in practice, to the nearest neighbourhood. Some of Madrid's are a few hundred metres out, beyond the area the polygons cover, and one is hundreds of kilometres away: a coordinate error in the source, which no neighbourhood should absorb.

So the join takes a documented **tolerance**: a listing outside every polygon gets the nearest neighbourhood if it is within 0.001° (about 100 m), and none otherwise. A tolerance is a decision, and writing it down as a named constant makes it visible and easy to change.

In [16]:
GAP_TOLERANCE_DEGREES = 0.001

gap_positions, gap_neighbourhood_positions = spatial_index.query_nearest(
    listing_points[unmatched_positions], max_distance=GAP_TOLERANCE_DEGREES, all_matches=False)
matches = pd.DataFrame({
    "listing": np.concatenate([point_positions, unmatched_positions[gap_positions]]),
    "neighbourhood_position": np.concatenate([neighbourhood_positions, gap_neighbourhood_positions]),
})
matches["neighbourhood_id"] = neighbourhoods["LOCATIONID"].to_numpy()[matches["neighbourhood_position"]]
matches["neighbourhood"] = neighbourhoods["LOCATIONNAME"].to_numpy()[matches["neighbourhood_position"]]
first_match = matches.sort_values("neighbourhood_id").drop_duplicates("listing").set_index("listing")
listings["neighbourhood_id"] = first_match["neighbourhood_id"].reindex(listings.index).astype("str")
listings["neighbourhood"] = first_match["neighbourhood"].reindex(listings.index).astype("str")
listings.groupby("city", sort=False).agg(neighbourhoods=("neighbourhood", "nunique"),
                                         listings_without_one=("neighbourhood", lambda names: names.isna().sum()))

,neighbourhoods,listings_without_one
city,,
madrid,135,31
barcelona,69,0
valencia,73,0


The neighbourhood counts agree with the documentation: 135 in Madrid, 69 in Barcelona and 73 in Valencia, so every neighbourhood holds at least one listing. The listings left without one are more than about 100 m outside every polygon; the quality report below counts them.

### Checking the source's own enrichment

The distance columns are themselves an enrichment, made by the authors from the points of interest. Recomputing them is the best test of what they mean.

The distance to the city centre is a *great-circle* distance: the length of the shortest path between two points on the Earth's surface, taken as a sphere. The *haversine* formula computes it from the two points' latitudes $\varphi$ and longitudes $\lambda$, in radians:

$$d = 2R \arcsin \sqrt{\sin^2 \frac{\Delta\varphi}{2} + \cos\varphi_1 \cos\varphi_2 \sin^2 \frac{\Delta\lambda}{2}}$$

where $R$ is the Earth's mean radius, 6,371.0088 km. The second point may be an array, so that one call measures the distance from the centre to every listing:

In [17]:
EARTH_RADIUS_KM = 6371.0088  # mean radius of the Earth (IUGG)


def haversine_km(lat1: float, lon1: float, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    lat1, lon1, lat2, lon2 = (np.radians(degrees) for degrees in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

The distance to the metro is the distance to the nearest station, found with a `BallTree`: a spatial index for nearest-neighbour searches that supports haversine distances. It works in radians and returns distances as angles, which the Earth's radius turns into kilometres.

In [18]:
from sklearn.neighbors import BallTree

recomputed_difference_m = {}
for city in CITIES:
    city_listings = listings[listings["city"] == city.lower()]
    centre = pois_by_city[city]["City_Center"].iloc[0]
    centre_km = haversine_km(centre["Lat"], centre["Lon"],
                             city_listings["latitude"].to_numpy(), city_listings["longitude"].to_numpy())
    stations = BallTree(np.radians(pois_by_city[city]["Metro"][["Lat", "Lon"]].to_numpy()), metric="haversine")
    nearest_station_radians, _ = stations.query(np.radians(city_listings[["latitude", "longitude"]].to_numpy()), k=1)
    recomputed_difference_m[city] = {
        "centre: median difference": np.median(centre_km - city_listings["dist_city_centre_km"]) * 1000,
        "centre: 95% within": np.quantile(np.abs(centre_km - city_listings["dist_city_centre_km"]), 0.95) * 1000,
        "metro: median difference": np.median(nearest_station_radians[:, 0] * EARTH_RADIUS_KM
                                              - city_listings["dist_metro_km"]) * 1000,
    }
pd.DataFrame(recomputed_difference_m).round(1)

,Madrid,Barcelona,Valencia
centre: median difference,-4.6,-2.9,-2.4
centre: 95% within,10.3,6.6,4.2
metro: median difference,-0.4,-0.3,-0.6


The recomputed distances agree with the source to within metres (the table is in metres), so the columns mean what the documentation says: the straight-line distance to the city centre, and to the nearest metro station. The small systematic difference is most likely a different way of computing distances on the Earth's surface, irrelevant at this scale. The landmark distances are distances to a line rather than to a point, and are not recomputed here.

## Data-quality checks

A quality rule is a condition every row should meet. Some come from the documentation (prices rounded to €1,000), some from the domain (no home was built after 2018 in a 2018 listing), some from the structure of the table (an asset should not change quarter). Each rule below is a boolean mask of the rows that break it.

In [19]:
asset_rows = listings.groupby(["city", "asset_id"])["period"]
quality_rules = {
    "price not a multiple of €1,000": listings["price_eur"] % 1000 != 0,
    "built area below 20 m² or above 500 m²": ~listings["built_area_m2"].between(20, 500),
    "construction year before 1500 or after 2018": ~listings["construction_year"].between(1500, 2018)
                                                   & listings["construction_year"].notna(),
    "more than 10 bedrooms": listings["bedrooms"] > 10,
    "more than 10 bathrooms": listings["bathrooms"] > 10,
    "no bedroom, but not a studio": (listings["bedrooms"] == 0) & (listings["is_studio"] == 0),
    "outside every neighbourhood": listings["neighbourhood"].isna(),
    "asset repeated within its quarter": listings.duplicated(["city", "asset_id", "period"], keep=False),
    "asset in more than one quarter": asset_rows.transform("nunique") > 1,
    "parking flags disagree": listings["has_parking_space"] != listings["parking_included_in_price"],
}
quality_report = pd.DataFrame({
    "rows": {rule: int(broken.sum()) for rule, broken in quality_rules.items()},
    "share (%)": {rule: round(100 * float(broken.mean()), 2) for rule, broken in quality_rules.items()},
})
quality_report

,rows,share (%)
"price not a multiple of €1,000",0,0.00
built area below 20 m² or above 500 m²,324,0.17
construction year before 1500 or after 2018,349,0.18
more than 10 bedrooms,81,0.04
more than 10 bathrooms,28,0.01
"no bedroom, but not a studio",290,0.15
outside every neighbourhood,31,0.02
asset repeated within its quarter,68573,36.11
asset in more than one quarter,0,0.00
parking flags disagree,0,0.00


The rules fall into three groups:

- **Rules that hold everywhere** confirm what the documentation says: every price is a multiple of €1,000, no asset appears in more than one quarter, and the two parking flags never disagree (they are the same variable twice).
- **Rules broken by a few rows** point at entry errors or at unusual properties: construction years like 1 or 2291, flats with more than ten bedrooms, homes of more than 500 m², a listing hundreds of kilometres from its city.
- **One rule is broken by many rows:** more than a third of the rows belong to an asset that appears more than once *within the same quarter*. That one deserves a closer look.

In [20]:
repeated = listings[quality_rules["asset repeated within its quarter"]]
repetitions = repeated.groupby(["city", "asset_id", "period"])
pd.DataFrame({
    "copies": repetitions.size().describe(percentiles=[0.5, 0.9]),
    "price spread (%)": (100 * repetitions["price_eur"].agg(lambda prices: prices.max() / prices.min() - 1)
                         ).describe(percentiles=[0.5, 0.9]),
    "north-south spread (m)": (repetitions["latitude"].agg(lambda latitudes: latitudes.max() - latitudes.min())
                               * METRES_PER_DEGREE).describe(percentiles=[0.5, 0.9]),
}).loc[["count", "50%", "90%", "max"]].round(1)

,copies,price spread (%),north-south spread (m)
count,28574.0,28574.0,28574.0
50%,2.0,4.5,42.9
90%,3.0,14.6,85.1
max,11.0,447.4,15391.8


Most repeated assets appear two or three times in the same quarter, and the copies never share exactly the same price and location. The anonymisation explains much of that: it moved every copy 30 to 60 m independently, and multiplied every price by its own factor between 0.975 and 1.025, which can separate two copies of one price by at most about 5% (1.025 / 0.975), plus the rounding to €1,000. About half of the groups stay within that. One group in ten, though, has prices more than 14% apart, which the noise cannot produce: those copies differ in the source itself, for instance because the price changed while the ad was online. The largest spreads (prices several times apart, copies kilometres apart) can only be different dwellings under one identifier. The documentation calls `ASSETID` a unique identifier of the asset; in this data it is not unique even within a quarter.

The pipeline **reports** these rows and keeps them unchanged. Whether a 2291 construction year should become a missing value, or a repeated asset should be collapsed to one row, depends on what the data is used for; an ETL that silently "fixed" them would take that decision away from every user. The report travels with the table.

## Load

The table is written as **Parquet**, a columnar file format: it stores each column's type with the data (the nullable integers, the categories and the strings come back exactly as they were), compresses well, and lets a reader load only the columns it needs. A CSV keeps no types, so every reader would have to guess them again.

In [21]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
listings.to_parquet(OUTPUT_PARQUET, index=False)
quality_report.to_csv(DATA_DIR / "idealista18_quality_report.csv")
print(f"{OUTPUT_PARQUET}: {OUTPUT_PARQUET.stat().st_size / 1e6:.1f} MB, {len(listings):,} rows, {listings.shape[1]} columns")

../../data/idealista18_listings.parquet: 15.3 MB, 189,923 rows, 44 columns


Reading the file back must give exactly the same table, values and types:

In [22]:
pd.testing.assert_frame_equal(pd.read_parquet(OUTPUT_PARQUET), listings)
listings.dtypes

city                                   str
asset_id                               str
period                               Int32
price_eur                          float64
unit_price_eur_m2                  float64
built_area_m2                        Int32
bedrooms                             Int32
bathrooms                            Int32
has_terrace                          Int32
has_lift                             Int32
has_air_conditioning                 Int32
has_parking_space                    Int32
parking_included_in_price            Int32
parking_space_price_eur            Float64
faces_north                          Int32
faces_south                          Int32
faces_east                           Int32
faces_west                           Int32
has_box_room                         Int32
has_wardrobe                         Int32
has_swimming_pool                    Int32
has_doorman                          Int32
has_garden                           Int32
is_duplex  